In [62]:
import soundfile as sf
import io
import numpy as np
import base64
import requests
import subprocess
import librosa
from IPython.display import Audio

In [63]:
SAMPLE_RATE = 16000

In [64]:
def compress_to_opus(bytes: bytes) -> bytes:
  process = subprocess.Popen(
    ["ffmpeg", "-i", "pipe:0", "-c:a", "libopus", "-f", "opus", "pipe:1"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE # subprocess.DEVNULL 하면 속도 조금 더 빨라짐
  )

  out, err = process.communicate(input=bytes)
  return out, err 

In [65]:
def np_to_wav(audio: np.ndarray, sample_rate:int) -> bytes:
  buffer = io.BytesIO()
  sf.write(buffer, audio, sample_rate, format='wav')
  return buffer.getvalue()

In [66]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [67]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=4000, scale=400))
  rand_len = np.clip(rand_len, 3500, 4500)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [68]:
idx = 0

In [69]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [70]:
output = []
for segment in segments:
  audio = np_to_wav(segment, SAMPLE_RATE)
  audio, _ = compress_to_opus(audio)
  audio_b64 = base64.b64encode(audio).decode('utf-8')
  params = {
    "group": "1", 
    "user": "1", 
    "audio": audio_b64
  }

  res = requests.get("https://api.sangjeong.com:8080/whisper/stt_duration", params=params)
  # print(res)
  d = res.json()
  print(d)
  output = [ *output, *d['completed'] ]

{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': [{'start': 0.0, 'end': 0.72, 'text': ' BBC', 'lang': 'ko'}, {'start': 0.72, 'end': 1.4, 'text': ' 생방송', 'lang': 'ko'}, {'start': 1.4, 'end': 1.92, 'text': ' 인터뷰도', 'lang': 'ko'}]}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': [{'start': 0.0, 'end': 0.72, 'text': ' BBC', 'lang': 'ko'}, {'start': 0.72, 'end': 1.4, 'text': ' 생방송', 'lang': 'ko'}, {'start': 1.4, 'end': 1.92, 'text': ' 인터뷰도', 'lang': 'ko'}, {'start': 1.88, 'en

In [74]:
for s in output:
  print(s["text"])
  print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")

BBC 생방송 인터뷰도 중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
	 start:0.0, end:7.72575
유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 등장했습니다.
	 start:7.82525, end:13.2130625
배영진 기자입니다.
	 start:13.653062499999999, end:14.6364375
BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리 부산대 교수, 유튜브 영상 조회수가,600만 건이 넘는데 켈리 교수 가족은 세계적인 유명인사가 됐습니다.
	 start:16.1461875, end:29.851812499999998
언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
	 start:30.9591875, end:35.0794375
춤을 췄던 첫째 딸 메리아는 사탕을 물었고 예나 예나 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
	 start:35.0875, end:46.969562499999995
disaster. 
	 start:48.187124999999995, end:48.62712499999999
I immediately called texted texted or emailed the BBC. 
	 start:49.08712499999999, end:52.307125
I communicated with the BBC immediately afterwards I apologized to them. 
	 start:52.408312499999994, end:55.7483125
I said that if they never us back or never asked me to be on television again, I would understand.
	 start:55.80831249999999, end:60.751749999999994
귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다. 
	 start:61.330999999999